# Lektion 19: 🏁 Das große System – live getestet



| | |
|---|---|
| **Zeit** | ~75 Minuten |
| **Schwierigkeit** | Fortgeschritten |
| **Voraussetzungen** | Lektion 01–18 |
| **Du baust** | Den kompletten End-to-End-Test deines finalen Assistenten – und den Start der echten Web-Oberfläche |

## 🎯 Nach dieser Lektion kannst du:

- das finale System (final-project/) über seine API komplett durchtesten

- alle Lektionen im Code des Finale wiedererkennen (Code-Recognition)

- den Human-in-the-Loop-Roundtrip über HTTP nachweisen

- die finale Streamlit-Oberfläche starten

- erklären, wie Notebook-Wissen und finale Anwendung zusammenhängen

## ⚙️ Setup – einmal pro Notebook ausführen

Diese Zelle findet die Projektwurzel und lädt die **zentrale KI-Konfiguration** aus deiner `.env` (siehe `docs/local-ai-server.md`). Du musst nie Notebook-Code ändern, um den Anbieter zu wechseln.

In [1]:
import sys
from pathlib import Path

# Projektwurzel finden (dort liegt der Ordner "lernakademie")
ROOT = Path.cwd()
while not (ROOT / "lernakademie").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "lernakademie").exists():
    raise RuntimeError("Projektwurzel nicht gefunden – starte das Notebook im Repo-Ordner.")
sys.path.insert(0, str(ROOT))

from lernakademie.config import llm_zusammenfassung

_info = llm_zusammenfassung()  # zeigt NIE den API-Key selbst
print("KI-Konfiguration:", _info["anbieter"], "| Modell:", _info["modell"],
      "| Key:", _info["api_key"])
if _info["anbieter"] == "offline":
    print("Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).")


KI-Konfiguration: offline | Modell: offline-simulation | Key: nicht gesetzt (optional)
Offline-Modus: alles laeuft – Antworten sind Simulationen ([OFFLINE]).


## 🏁 Das Finale – was du gebaut hast

Im Ordner `final-project/knowledge-research-agent/` läuft **dein gesamtes Wissen** als echtes System:

```text
Browser (Streamlit) → FastAPI-Backend → LangGraph-Workflow
                                        ├─ Vektor-RAG (Lektion 08/09)
                                        ├─ Knowledge Graph (Lektion 10-12)
                                        └─ Freigabe-Workflow (Lektion 15/17)
```

**Warum über die API testen?** Der TestClient spricht **exakt dieselben Endpunkte** an wie das Frontend – du testest damit das System, wie der Browser es sieht. Und: alles läuft offline (Simulation), wie gewohnt.

In [2]:
import sys
from pathlib import Path

FINALE = ROOT / "final-project" / "knowledge-research-agent"
sys.path.insert(0, str(FINALE))

from fastapi.testclient import TestClient
from backend.api.main import app

client = TestClient(app)
status = client.get("/status").json()
print("Status:", {k: status[k] for k in ("status", "modus", "anbieter", "modell", "api_key")})
print("Dokumente:", status["dokumente"], "| Graph:", status["graph_knoten"], "Nodes,", status["graph_kanten"], "Kanten")

Status: {'status': 'ok', 'modus': 'offline', 'anbieter': 'Offline-Modus (Simulation)', 'modell': 'offline-simulation', 'api_key': 'nicht gesetzt (optional)'}
Dokumente: 2 | Graph: 8 Nodes, 9 Kanten


### 🔎 Was sehen wir hier?
Der Status-Endpunkt zeigt dir **ohne Geheimnisse**: Anbieter, Modell, Key-Status („gesetzt"/„nicht gesetzt") – genau wie dein Verbindungstest aus START_HERE. Die zentrale Konfiguration durchzieht ALLES.

## Test 1: Chat mit Routing (dein Build-3-Agent!)

In [3]:
def frage_system(text):
    antwort = client.post("/chat", json={"nachricht": text, "session_id": "notebook-test"})
    antwort.raise_for_status()
    return antwort.json()

for frage in ("Was steht ueber die Abteilung Engineering?",
              "Wer haengt am AI Assistant dran?",
              "Was wissen wir gesamt ueber das Projekt?"):
    daten = frage_system(frage)
    print(f"FRAGE: {frage}")
    print(f"  Route: {daten['route']} | Doku: {len(daten['dokumente'])} | Graph: {len(daten['graph_fakten'])}")
    for fakt in daten["graph_fakten"][:2]:
        print("   Graph-Fakt:", fakt)
    print()

FRAGE: Was steht ueber die Abteilung Engineering?
  Route: vektor | Doku: 2 | Graph: 0

FRAGE: Wer haengt am AI Assistant dran?
  Route: graph | Doku: 0 | Graph: 4
   Graph-Fakt: Engineering --WORKS_ON--> AI Assistant
   Graph-Fakt: Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant

FRAGE: Was wissen wir gesamt ueber das Projekt?
  Route: hybrid | Doku: 2 | Graph: 4
   Graph-Fakt: Engineering --WORKS_ON--> AI Assistant
   Graph-Fakt: Data & Analytics --PROVIDES_DATA_FOR--> AI Assistant



## Test 2: Dokumente hochladen + Verlauf

Dein Assistent lernt zur Laufzeit dazu – Wissensbasis-Verwaltung wie im Frontend:

In [4]:
upload = client.post("/documents", json={
    "titel": "notiz-aus-notebook",
    "inhalt": "Dies ist eine Testnotiz ueber die Weiterbildung im Unternehmen. "
              "Alle Mitarbeitenden haben Zugang zum Lernportal."})
print("Upload:", upload.json())

historie = client.get("/history", params={"session_id": "notebook-test"}).json()
print("Verlauf-Eintraege:", len(historie["eintraege"]))

Upload: {'titel': 'notiz-aus-notebook', 'neue_chunks': 1}
Verlauf-Eintraege: 3


## Test 3: Der Freigabe-Roundtrip über HTTP

Jetzt der stärkste Beweis: **Vorschlag → Interrupt → Genehmigung → Graph verändert.** Genau wie in Lektion 15 – nur dass der Mensch hier "ein Browser" wäre:

In [5]:
kanten_vorher = client.get("/status").json()["graph_kanten"]

vorschlag = client.post("/facts/propose", json={
    "von": "Clara", "typ": "WORKS_FOR", "zu": "Acme",
    "grund": "Steht implizit im Firmenprofil."}).json()
print("Status:", vorschlag["status"])
print("Interrupt-Frage:", vorschlag["details"]["frage"])
print("Fakt:", vorschlag["details"]["fakt"])

freigabe = client.post("/facts/approve", json={
    "thread_id": vorschlag["thread_id"], "genehmigt": True}).json()
print("Ergebnis:", freigabe["ergebnis"])

kanten_nachher = client.get("/status").json()["graph_kanten"]
print(f"Kanten: {kanten_vorher} -> {kanten_nachher} (+1 = Genehmigung wirkt)")

Status: wartet_auf_freigabe
Interrupt-Frage: Soll dieser Fakt in den Knowledge Graph eingetragen werden?
Fakt: Clara --WORKS_FOR--> Acme
AUDIT: GENEHMIGT: Clara --WORKS_FOR--> Acme
Ergebnis: GENEHMIGT: Clara --WORKS_FOR--> Acme
Kanten: 9 -> 10 (+1 = Genehmigung wirkt)


## ✏️ Deine Aufgabe: Der Ablehnungs-Beweis (und dein M6)

Du beweist, dass Ablehnung den Graph NICHT verändert – und erkennst dein eigenes Wissen im Finale-Code wieder.

**Anforderungen:**

- Schlage einen **riskanten** Fakt vor (z. B. `Ben MANAGES Acme`, Grund: „Vermutung ohne Beleg") und lehne ihn ab (`genehmigt=False`).

- Beweis: Die Kantenanzahl ist danach unverändert, der `ABGELEHNT`-Eintrag steht im Audit-Log (`/audit`).

- **Code-Recognition:** Öffne `final-project/knowledge-research-agent/backend/graphs/workflow.py` und `backend/agents/approval.py`. Markiere (als Kommentar hier): mindestens 4 Wiedererkennungen aus deinen Lektionen (z. B. „klassifizieren = Lektion 13", „interrupt = Lektion 15/17").

**Schreibe deinen Code in die folgende Zelle:**

In [6]:
kanten_vor_ablehnung = client.get("/status").json()["graph_kanten"]

vorschlag = client.post("/facts/propose", json={
    "von": "Ben", "typ": "MANAGES", "zu": "Acme",
    "grund": "Vermutung ohne Beleg."}).json()
freigabe = client.post("/facts/approve", json={
    "thread_id": vorschlag["thread_id"], "genehmigt": False}).json()
print("Ergebnis:", freigabe["ergebnis"])

kanten_nach_ablehnung = client.get("/status").json()["graph_kanten"]
print("Kanten unverändert:", kanten_vor_ablehnung == kanten_nach_ablehnung)
print("Audit:", [e for e in client.get("/audit").json()["eintraege"] if "ABGELEHNT" in e])

# MEINE WIEDERERKENNUNGEN:
# 1. workflow.py: klassifiziere() = meine Router-Logik aus Lektion 13/15
# 2. workflow.py: konditionale Kanten = die Weiche (Lektion 13)
# 3. approval.py: interrupt() vor der Aenderung = Lektion 15 (Freigabe)
# 4. approval.py: AUDIT_LOG = mein Audit aus Lektion 15
# 5. api/main.py: Pydantic-Schemas = strukturierte Ausgaben (Lektion 09)
# 6. retrieval/vector.py: Retriever = Build 2 (Lektion 08/09)

AUDIT: ABGELEHNT: Ben --MANAGES--> Acme
Ergebnis: ABGELEHNT: Ben --MANAGES--> Acme
Kanten unverändert: True
Audit: ['ABGELEHNT: Ben --MANAGES--> Acme']


**💡 Warum diese Loesung funktioniert:**

Der Ablehnungs-Beweis schließt den Sicherheits-Kreis: Lesen erlaubt der Agent allein (Whitelist, Lektion 12), Schreiben nur mit dir (Lektion 15/17) – und das Audit-Log macht jede Entscheidung nachvollziehbar. **Du hast dieses System nicht nur benutzt – du hast jeden seiner Bausteine selbst gebaut.**

### ✅ Richtig, wenn:

- [ ] Die Kantenanzahl ist nach der Ablehnung unverändert.

- [ ] Das Audit-Log enthält den ABGELEHNT-Eintrag.

- [ ] Mindestens 4 Wiedererkennungen sind als Kommentar dokumentiert.

**🧪 Fuehre die folgende Zelle aus, um dein Ergebnis zu pruefen:**

In [7]:
audit = client.get("/audit").json()["eintraege"]
assert any("ABGELEHNT" in e for e in audit), "Ablehnung muss im Audit-Log stehen!"
assert client.get("/status").json()["graph_kanten"] == kanten_nachher
print("Alle Prüfungen bestanden! ✅")

Alle Prüfungen bestanden! ✅


<details><summary>💡 Hinweis 1 (oeffnen, wenn du nicht weiterkommst)</summary>

Vorschlag und Freigabe laufen exakt wie in Test 3 – nur mit `genehmigt=False`.
</details>

<details><summary>💡 Hinweis 2 (oeffnen, wenn du nicht weiterkommst)</summary>

Audit: `client.get("/audit").json()["eintraegen"]`.
</details>

<details><summary>💡 Hinweis 3 (oeffnen, wenn du nicht weiterkommst)</summary>

Für die Wiedererkennungen: Lies die Dateien mit `Path(...).read_text()` oder öffne sie im Editor.
</details>

## 📝 Reflexion

Beantworte in deinen eigenen Worten:

1. Vergleiche mit deinem Build-1-Assistenten aus Lektion 05: Was von deinem damaligen Code steckt konzeptionell noch in diesem System?

2. Warum sitzt der Verlauf im Backend (Sessions) und nicht im Frontend?

## 🖥️ Die finale Oberfläche starten

Jetzt der Moment: **Dein System als echte Web-App.** Zwei Terminals im Projektordner:

```bash
# Terminal 1 – Backend:
cd final-project/knowledge-research-agent
uvicorn backend.api.main:app --port 8000

# Terminal 2 – Frontend:
streamlit run final-project/knowledge-research-agent/frontend/app.py
```

Oder alles auf einmal (inkl. Neo4j, wenn Docker vorhanden): `cd final-project/knowledge-research-agent && docker compose up --build`

**Probiere im Browser** (http://localhost:8501):
- 💬 Chat mit Routen-Badges (vektor/graph/hybrid) und Quellen-Expandern
- 🕸️ Graph-Explorer: Nodes klicken, Properties und Verbindungen sehen
- 📄 Dokumente hochladen und darüber fragen
- 🧪 Prompt-Labor: System-Prompt und Temperature live verändern
- ✋ Freigaben: einen Fakt genehmigen UND ablehnen (Audit prüfen!)

Die Sidebar zeigt dir immer Anbieter, Modell und Endpoint – **niemals** den API-Key.

In [8]:
from pathlib import Path
for datei in ("frontend/app.py", "backend/api/main.py"):
    p = FINALE / datei
    print("Bereit:", datei, "|", p.exists())
print("\nStartbefehle stehen im Markdown oben. Viel Erfolg mit DEINEM System! 🚀")

Bereit: frontend/app.py | True
Bereit: backend/api/main.py | True

Startbefehle stehen im Markdown oben. Viel Erfolg mit DEINEM System! 🚀


## ➡️ Weiter

**Letzte Lektion:** `notebooks/11_testing/20_testing_und_evaluierung.ipynb` – du lernst, dein System zu prüfen wie eine Ingenieurin – und schreibst deine Abschluss-Reflexion.